In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import os

# ============================================================
# 1) Load MoneyPuck team game-by-game data
# ============================================================

url = "https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
raw_df = pd.read_csv(StringIO(response.text))

# ============================================================
# 2) Create game-situation dataset
#    We keep both regular season and playoffs
# ============================================================

final_game_situation_data = raw_df[
    (raw_df["season"] < 2025) &
    (raw_df["situation"].isin(["5on5", "5on4", "4on5"]))
].copy()

final_game_situation_data["season_type"] = final_game_situation_data["playoffGame"].map({
    0: "Regular season",
    1: "Playoffs"
})

final_game_situation_data = final_game_situation_data[[
    "season",
    "season_type",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "situation",
    "goalsFor",
    "goalsAgainst",
    "faceOffsWonFor",
    "faceOffsWonAgainst",
    "shotsOnGoalFor",
    "shotsOnGoalAgainst",
    "shotAttemptsFor",
    "shotAttemptsAgainst",
    "giveawaysFor",
    "giveawaysAgainst"
]].copy()

# ============================================================
# 3) Add situation-level calculated variables
# ============================================================

final_game_situation_data["goal_diff"] = (
    final_game_situation_data["goalsFor"] -
    final_game_situation_data["goalsAgainst"]
)

final_game_situation_data["total_faceoffs"] = (
    final_game_situation_data["faceOffsWonFor"] +
    final_game_situation_data["faceOffsWonAgainst"]
)

final_game_situation_data["faceoff_pct"] = np.where(
    final_game_situation_data["total_faceoffs"] > 0,
    final_game_situation_data["faceOffsWonFor"] /
    final_game_situation_data["total_faceoffs"],
    np.nan
)

final_game_situation_data["faceoff_pct_100"] = (
    final_game_situation_data["faceoff_pct"] * 100
)

# ============================================================
# 4) Create full-game result dataset
#    This is based on situation == "all"
# ============================================================

game_results = raw_df[
    (raw_df["season"] < 2025) &
    (raw_df["situation"] == "all")
].copy()

game_results["season_type"] = game_results["playoffGame"].map({
    0: "Regular season",
    1: "Playoffs"
})

game_results = game_results[[
    "gameId",
    "team",
    "season_type",
    "goalsFor",
    "goalsAgainst"
]].copy()

game_results = game_results.rename(columns={
    "goalsFor": "game_goals_for",
    "goalsAgainst": "game_goals_against"
})

game_results["win"] = (
    game_results["game_goals_for"] >
    game_results["game_goals_against"]
).astype(int)

game_results["draw"] = (
    game_results["game_goals_for"] ==
    game_results["game_goals_against"]
).astype(int)

game_results["lose"] = (
    game_results["game_goals_for"] <
    game_results["game_goals_against"]
).astype(int)

game_results["points"] = (
    game_results["win"] * 2 +
    game_results["draw"] * 1
)

game_results["game_goal_diff"] = (
    game_results["game_goals_for"] -
    game_results["game_goals_against"]
)

# ============================================================
# 5) Merge full-game results into situation-level dataset
# ============================================================

final_game_situation_data = final_game_situation_data.merge(
    game_results,
    on=["gameId", "team", "season_type"],
    how="left"
)

# ============================================================
# 6) Clean team codes
# ============================================================

team_code_map = {
    "L.A": "LAK",
    "N.J": "NJD",
    "S.J": "SJS",
    "T.B": "TBL"
}

final_game_situation_data["team_clean"] = (
    final_game_situation_data["team"].replace(team_code_map)
)

final_game_situation_data["opposingTeam_clean"] = (
    final_game_situation_data["opposingTeam"].replace(team_code_map)
)

# ============================================================
# 7) Save final dataset
# ============================================================

final_game_situation_data.to_csv(
    "final_game_situation_data.csv",
    index=False
)

final_game_situation_data.head()

,season,season_type,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,...,faceoff_pct_100,game_goals_for,game_goals_against,win,draw,lose,points,game_goal_diff,team_clean,opposingTeam_clean
0,2008,Regular season,2008020001,20081004,NYR,T.B,AWAY,5on5,1.0,1.0,...,51.351351,2.0,1.0,1,0,0,2,1.0,NYR,TBL
1,2008,Regular season,2008020001,20081004,NYR,T.B,AWAY,4on5,0.0,0.0,...,71.428571,2.0,1.0,1,0,0,2,1.0,NYR,TBL
2,2008,Regular season,2008020001,20081004,NYR,T.B,AWAY,5on4,1.0,0.0,...,35.294118,2.0,1.0,1,0,0,2,1.0,NYR,TBL
3,2008,Regular season,2008020003,20081005,NYR,T.B,HOME,5on5,1.0,1.0,...,32.258065,2.0,1.0,1,0,0,2,1.0,NYR,TBL
4,2008,Regular season,2008020003,20081005,NYR,T.B,HOME,4on5,0.0,0.0,...,28.571429,2.0,1.0,1,0,0,2,1.0,NYR,TBL


In [2]:
final_game_situation_data.shape

(130968, 31)

In [3]:
final_game_situation_data["season_type"].value_counts()

season_type
Regular season    121776
Playoffs            9192
Name: count, dtype: int64